# SPATIAL INTELLIGENCE ANALYSIS — DUBAI TOWER, RESIDENTIAL FLOOR PLAN (Level 04)

Single-floor visibility & graph analysis of **Level 04** (`assets/obj/F04.obj`).

#### Environment note — topologicpy is pip-installed (no `sys.path` patch needed)

In [ ]:
# topologicpy is pip-installed in this environment, so no sys.path hack is needed.
# (External path from the original tutorial machine has been removed.)

## 1. Import the needed libraries

In [3]:
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [4]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.50) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [5]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [6]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


#### Resolve file paths — locate the floor-plan OBJ and set the BREP export path

In [ ]:
from pathlib import Path

# === Floor selection =========================================================
FLOOR_TAG = "F04"          # Level 04   (the Level 03 notebook sets this to "F03")

# Resolve <repo>/assets/obj/<FLOOR_TAG>.obj robustly. When run from VS Code /
# Jupyter the notebook CWD is the DubaiTower folder, so we search it and its
# parents for assets/obj/<tag>.obj.
def _find_obj(tag):
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "assets" / "obj" / f"{tag}.obj"
        if p.exists():
            return p
    hits = list(Path.cwd().parent.glob(f"**/assets/obj/{tag}.obj"))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find assets/obj/{tag}.obj under the project root")

OBJ_PATH = _find_obj(FLOOR_TAG)

# Per-floor BREP export kept next to this notebook (one surface file per level).
BREP_PATH = Path.cwd() / "Asset" / f"{FLOOR_TAG}_export_surface.brep"
BREP_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"FLOOR_TAG : {FLOOR_TAG}")
print(f"OBJ_PATH  : {OBJ_PATH} | exists={OBJ_PATH.exists()}")
print(f"BREP_PATH : {BREP_PATH}")
if not OBJ_PATH.exists():
    raise FileNotFoundError(f"OBJ file not found: {OBJ_PATH}")

## 5. Import the gallery floor plan

In [11]:
objects = Topology.ByOBJPath(str(OBJ_PATH))
print(objects)

[<topologic_core.Cluster object at 0x0000024B516AC570>, <topologic_core.Cluster object at 0x0000024B5292FCB0>]


#### De-triangulate the mesh — dissolve the raw OBJ triangles into clean gallery faces (Shapely)

In [12]:
from shapely.geometry import Polygon as ShapelyPolygon, MultiPolygon
from shapely.ops      import unary_union

# -- helpers ---------------------------------------------------------------
def shapely_to_face(polygon):
    coords = list(polygon.exterior.coords)[:-1]
    if len(coords) < 3:
        return None
    outer = Wire.ByVertices(
        [Vertex.ByCoordinates(x, y, 0.0) for x, y in coords], close=True)
    if outer is None:
        return None
    holes = []
    for interior in polygon.interiors:
        hc = list(interior.coords)[:-1]
        if len(hc) < 3:
            continue
        hw = Wire.ByVertices(
            [Vertex.ByCoordinates(x, y, 0.0) for x, y in hc], close=True)
        if hw is not None:
            holes.append(hw)
    return Face.ByWires(outer, holes) if holes else Face.ByWire(outer)

def extract_polygons(geom):
    """Recursively extract only Polygon types; drop lines/points."""
    if geom.geom_type == "Polygon":
        return [geom] if geom.area > 0 else []
    elif geom.geom_type in ("MultiPolygon", "GeometryCollection"):
        result = []
        for g in geom.geoms:
            result.extend(extract_polygons(g))
        return result
    return []

# -- load and merge raw OBJ faces (detriangulate by dissolve) -------------
objects_raw = Topology.ByOBJPath(str(OBJ_PATH))
cluster_raw = Cluster.ByTopologies(objects_raw) if isinstance(objects_raw, list) else objects_raw
raw_faces   = Topology.Faces(cluster_raw) or []
print(f"Faces from OBJ: {len(raw_faces)}")

# -- build valid Shapely polygons ------------------------------------------
shapely_polys = []
for f in raw_faces:
    verts = Topology.Vertices(f)
    coords = [(v.X(), v.Y()) for v in verts]
    unique = list(set(coords))
    if len(unique) < 3:
        continue
    poly = ShapelyPolygon(coords)
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_valid and not poly.is_empty:
        shapely_polys.append(poly)

print(f"Valid Shapely polygons: {len(shapely_polys)}")

# -- dissolve ---------------------------------------------------------------
dissolved = unary_union(shapely_polys)
if not dissolved.is_valid:
    dissolved = dissolved.buffer(0)
print(f"Dissolved type: {dissolved.geom_type}")

# -- convert back to TopologicPy Face --------------------------------------
poly_list = extract_polygons(dissolved)
print(f"Extracted polygons: {len(poly_list)}")

if len(poly_list) == 0:
    print("ERROR: No valid polygons; check OBJ path and geometry.")
elif len(poly_list) == 1:
    raw = shapely_to_face(poly_list[0])
    gallery = Topology.RemoveCollinearEdges(raw) or raw
    print("Gallery (single face):", gallery)
else:
    parts = [shapely_to_face(p) for p in poly_list]
    parts = [f for f in parts if f is not None]
    parts = [(Topology.RemoveCollinearEdges(f) or f) for f in parts]
    gallery = Cluster.ByTopologies(parts) if len(parts) > 1 else parts[0]
    print(f"Gallery ({len(parts)} parts):", gallery)


Faces from OBJ: 1238
Valid Shapely polygons: 1238
Dissolved type: Polygon
Extracted polygons: 1
Gallery (single face): <topologic_core.Face object at 0x0000024B52A43E70>


#### Export the merged gallery to a BREP file

In [13]:
# Export merged (de-triangulated) gallery, not raw OBJ triangles
gallery_faces_for_export = Topology.Faces(gallery) or []
gallery_cluster = Cluster.ByTopologies(gallery_faces_for_export) if len(gallery_faces_for_export) > 1 else gallery
gallery_merged = Topology.SelfMerge(gallery_cluster, tolerance=0.001) or gallery_cluster

output_path = str(BREP_PATH)
status = Topology.ExportToBREP(gallery_merged, path=output_path, overwrite=True)
print(f"Export status: {status}")
print(f"BREP file saved to: {output_path}")

Topology.Faces - Warning: The input is a Face. Returning the same face embedded in a list.
caller name: <module>
Export status: True
BREP file saved to: c:\Users\MOHA9808\Downloads\New folder\GML.26\Asset\resi floor_export_surface.brep


#### Reload the gallery from the exported BREP

In [14]:
# Use the BREP generated in the previous step
gallery = Topology.ByBREPPath(output_path)

#### Pre-analysis mesh clean-up — remove collinear edges & merge coplanar faces

In [15]:
# Mesh clean-up before any analysis steps (TopologicPy de-triangulation pass)
gallery_faces = Topology.Faces(gallery) or []
gallery_faces = [f for f in gallery_faces if f]

if not gallery_faces:
    raise ValueError("Mesh cleaning failed: gallery has no valid faces.")

# Clean each face, then merge coplanar pieces to remove triangulation artifacts
clean_gallery_faces = []
for f in gallery_faces:
    cf = Topology.RemoveCollinearEdges(f) or f
    area = Face.Area(cf) if cf else 0
    if cf and area and area > 1e-6:
        clean_gallery_faces.append(cf)

if not clean_gallery_faces:
    raise ValueError("Mesh cleaning failed: no faces left after cleanup.")

clean_cluster = Cluster.ByTopologies(clean_gallery_faces) if len(clean_gallery_faces) > 1 else clean_gallery_faces[0]
merged_gallery = Topology.SelfMerge(clean_cluster, tolerance=0.001) or clean_cluster

gallery = merged_gallery
gallery_faces = Topology.Faces(gallery) or []
print(f"Pre-analysis clean mesh ready: {len(gallery_faces)} merged faces.")

Topology.Faces - Warning: The input is a Face. Returning the same face embedded in a list.
caller name: <module>
Topology.Faces - Warning: The input is a Face. Returning the same face embedded in a list.
caller name: <module>
Pre-analysis clean mesh ready: 1 merged faces.


## 6. Show the geometry

In [16]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

## 7. Create a grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

if width is None or length is None:
    raise ValueError("Bounding rectangle failed. Make sure gallery is loaded correctly before creating the grid.")

# Consultant speed profile for isovist-heavy workflow.
# fast: best runtime, balanced: default quality/runtime, high: slowest but densest
PERFORMANCE_PRESET = "fast"
PRESET_SCALE = {"fast": 0.70, "balanced": 0.50, "high": 0.35}
scale = PRESET_SCALE.get(PERFORMANCE_PRESET, 0.70)

# Larger step -> fewer cells/faces -> faster downstream isovist analysis
vertex_grid_step = 28 * scale

# --- Floor-splitting grid kept CONSISTENT with the multi-floor notebook ---------
# edge_grid_step is the grid that SLICES the floor into analysis cells (faces).
# Keep it equal to NB_DubaiTower_MultiFloor_Spatial_Intelligence's GRID_SIZE so both
# analyses resolve the floor at the same resolution. (Preset value was 3*scale = 2.1.)
# If isovists get slow, raise ANALYSIS_GRID here AND GRID_SIZE in the multi-floor
# notebook together so they stay matched.
ANALYSIS_GRID = 1.0
edge_grid_step = ANALYSIS_GRID

# Safety caps prevent accidental very dense grids on large models
MAX_VERTEX_DIVS = 120
MAX_EDGE_DIVS = 220

u_div1 = min(MAX_VERTEX_DIVS, int(width / vertex_grid_step) + 2)
v_div1 = min(MAX_VERTEX_DIVS, int(length / vertex_grid_step) + 2)
u_div2 = min(MAX_EDGE_DIVS, int(width / edge_grid_step) + 2)
v_div2 = min(MAX_EDGE_DIVS, int(length / edge_grid_step) + 2)

uRange1 = [i * vertex_grid_step for i in range(u_div1)]
vRange1 = [i * vertex_grid_step for i in range(v_div1)]
uRange2 = [i * edge_grid_step for i in range(u_div2)]
vRange2 = [i * edge_grid_step for i in range(v_div2)]

grid1 = Grid.VerticesByDistances(gallery, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange2, vRange=vRange2)

print(
    f"Grid preset: {PERFORMANCE_PRESET} | vertex_step={vertex_grid_step:.2f} | edge_step={edge_grid_step:.2f} "
    f"| vertex_divs=({u_div1},{v_div1}) | edge_divs=({u_div2},{v_div2})"
)

## 8. Slice the floor plan with the edge grid to create a topologic shell

In [18]:
# Ensure we have a valid edge grid before slicing
if grid2 is None:
    b_r = Wire.BoundingRectangle(gallery)
    b_face = Face.ByWire(b_r) if b_r else None
    d_bbox = Topology.Dictionary(b_r) if b_r else None
    width_fallback = Dictionary.ValueAtKey(d_bbox, "width") if d_bbox else None
    length_fallback = Dictionary.ValueAtKey(d_bbox, "length") if d_bbox else None

    if b_face and width_fallback and length_fallback:
        uRange2 = [i * edge_grid_step for i in range(int(width_fallback / edge_grid_step) + 2)]
        vRange2 = [i * edge_grid_step for i in range(int(length_fallback / edge_grid_step) + 2)]
        grid2 = Grid.EdgesByDistances(b_face, clip=False, uRange=uRange2, vRange=vRange2)

if grid2 is None:
    raise ValueError("grid2 is invalid. Rerun section 7 (grid creation) before slicing.")

shell = Topology.Slice(gallery, grid2)
if shell is None:
    raise ValueError("Slicing failed. Check that both gallery and grid2 are valid topologies.")

faces = Topology.Faces(shell) or []
if not faces:
    raise ValueError("No faces were generated from the sliced shell.")

# Assign sequential unique face ids to each face and store the updated faces
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    faces[i] = Topology.SetDictionary(f, d)

print(f"Shell generated successfully with {len(faces)} faces.")

Shell generated successfully with 611 faces.


## 9. Derive analysis graphs from the shell

In [19]:
# Clean mesh before graph analysis: drop invalid faces and rebuild a consistent topology
raw_faces = Topology.Faces(shell) or []
valid_faces = [f for f in raw_faces if f]

if not valid_faces:
    raise ValueError("Mesh cleaning failed: no valid faces found in shell.")

# Reassign ids to the cleaned face list to keep indexing contiguous
for i, f in enumerate(valid_faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    valid_faces[i] = Topology.SetDictionary(f, d)

clean_shell = Cluster.ByTopologies(valid_faces)
if clean_shell is None:
    raise ValueError("Mesh cleaning failed: could not build cleaned topology from valid faces.")

faces = valid_faces
shell = clean_shell
print(f"Clean mesh ready: {len(faces)} faces.")

Clean mesh ready: 611 faces.


#### Build the analysis graph from the shell

In [20]:
# Note: Graph nodes automatically inherit entity dictionaries (including face_id)
analysis_graph = Graph.ByTopology(shell)

## 10. Derive and store the graph vertices and isovist vertices

> You can run automatic viewpoints (face centroids) OR manual viewpoints.
> Manual options:
> 1) Enter model coordinates directly as `(x, y)`.
> 2) Enter pixel coordinates from your red-dot image and map them to model space.

In [52]:
g_verts = Graph.Vertices(analysis_graph)

# ------------------------------
# MANUAL VIEWPOINT CONFIGURATION
# ------------------------------
USE_MANUAL_VIEWPOINTS = True

# Option A: direct model-space coordinates (x, y)
MANUAL_ISOVIST_POINTS_XY = [
    # Example: (120.0, 85.0),
    # Example: (95.0, 140.0),
]

# Option B: pixel coordinates from your red-dot image (px, py)
# Loaded from your shared marked image (approximate centers of red dots).
MANUAL_ISOVIST_PIXELS = [
    (257, 70),
    (229, 95),
    (92, 118),
    (186, 150),
    (229, 172),
    (123, 177),
    (122, 290),
    (182, 295),
    (227, 293),
    (92, 447),
    (170, 462),
    (230, 446),
    (258, 503),
    (100, 520),
]
IMAGE_SIZE_PX = (406, 611)  # (width_px, height_px) of your marked image
IMAGE_Y_AXIS_DOWN = True       # True for normal image coordinates (origin at top-left)

# Auto-augment manual points so they cover most of the floor plan.
AUTO_AUGMENT_MANUAL_POINTS = True
TARGET_MANUAL_POINT_COUNT = 40

def _as_bool(x):
    if isinstance(x, list):
        return any(bool(v) for v in x)
    return bool(x)

def _pixels_to_model(px, py, xmin, xmax, ymin, ymax, image_w, image_h, y_axis_down=True):
    if image_w <= 0 or image_h <= 0:
        raise ValueError("IMAGE_SIZE_PX must be positive.")
    u = max(0.0, min(1.0, px / image_w))
    v = max(0.0, min(1.0, py / image_h))
    x = xmin + u * (xmax - xmin)
    if y_axis_down:
        y = ymax - v * (ymax - ymin)
    else:
        y = ymin + v * (ymax - ymin)
    return (x, y)

# Build (host_face, centroid) pairs so each isovist starts from its parent face.
face_centroid_pairs = []
for f in faces:
    if not f:
        continue
    c = Topology.Centroid(f)
    if c:
        face_centroid_pairs.append((f, c))

# Build gallery face list for inside checks and host assignment in manual mode.
gallery_type = Topology.TypeAsString(gallery) if gallery else None
if gallery_type == "Face":
    gallery_faces = [gallery]
else:
    gallery_faces = [f for f in (Topology.Faces(gallery) or []) if f]

if not gallery_faces:
    raise ValueError("No valid gallery faces found.")

def _inside_any_gallery_face(v):
    for gf in gallery_faces:
        if _as_bool(Vertex.IsInternal2D(v, gf)):
            return True
    return False

def _nearest_host_face(v, candidate_faces):
    best_face = None
    best_d2 = None
    vx, vy = Vertex.X(v), Vertex.Y(v)
    for f in candidate_faces:
        c = Topology.Centroid(f)
        if not c:
            continue
        dx = vx - Vertex.X(c)
        dy = vy - Vertex.Y(c)
        d2 = dx * dx + dy * dy
        if best_d2 is None or d2 < best_d2:
            best_d2 = d2
            best_face = f
    return best_face

manual_vertices = []
if USE_MANUAL_VIEWPOINTS:
    # A) direct model XY points
    for x, y in MANUAL_ISOVIST_POINTS_XY:
        v = Vertex.ByCoordinates(x, y, 0.0)
        if _inside_any_gallery_face(v):
            manual_vertices.append(v)

    # B) image pixel points -> model XY points
    if MANUAL_ISOVIST_PIXELS:
        b_r_local = Wire.BoundingRectangle(gallery)
        d_local = Topology.Dictionary(b_r_local)
        xmin_local = Dictionary.ValueAtKey(d_local, "xmin")
        xmax_local = Dictionary.ValueAtKey(d_local, "xmax")
        ymin_local = Dictionary.ValueAtKey(d_local, "ymin")
        ymax_local = Dictionary.ValueAtKey(d_local, "ymax")
        img_w, img_h = IMAGE_SIZE_PX

        for px, py in MANUAL_ISOVIST_PIXELS:
            mx, my = _pixels_to_model(
                px, py, xmin_local, xmax_local, ymin_local, ymax_local, img_w, img_h, IMAGE_Y_AXIS_DOWN
            )
            v = Vertex.ByCoordinates(mx, my, 0.0)
            if _inside_any_gallery_face(v):
                manual_vertices.append(v)

if USE_MANUAL_VIEWPOINTS and manual_vertices:
    # Deduplicate manual points by rounded XY
    seen = set()
    unique_manual = []
    for v in manual_vertices:
        key = (round(Vertex.X(v), 6), round(Vertex.Y(v), 6))
        if key not in seen:
            seen.add(key)
            unique_manual.append(v)
    manual_vertices = unique_manual

    # Auto-augment with distributed face centroids for broader floor coverage.
    if AUTO_AUGMENT_MANUAL_POINTS and len(manual_vertices) < TARGET_MANUAL_POINT_COUNT:
        needed = TARGET_MANUAL_POINT_COUNT - len(manual_vertices)
        centroid_candidates = [p[1] for p in face_centroid_pairs if p[1] is not None]
        stride = max(1, len(centroid_candidates) // max(1, needed))
        added = 0
        for cv in centroid_candidates[::stride]:
            key = (round(Vertex.X(cv), 6), round(Vertex.Y(cv), 6))
            if key in seen:
                continue
            if not _inside_any_gallery_face(cv):
                continue
            seen.add(key)
            manual_vertices.append(cv)
            added += 1
            if len(manual_vertices) >= TARGET_MANUAL_POINT_COUNT:
                break
        print(f"Manual point augmentation added: {added}")

    iso_verts = manual_vertices
    iso_host_faces = [_nearest_host_face(v, faces) for v in iso_verts]
    print(f"Using manual viewpoints: {len(iso_verts)}")
else:
    # Runtime profile for automatic centroid sampling
    VIEWPOINT_CAP_BY_PRESET = {"fast": 220, "balanced": 360, "high": 600}
    MAX_ISOVIST_VIEWPOINTS = VIEWPOINT_CAP_BY_PRESET.get(PERFORMANCE_PRESET, 360)

    if len(face_centroid_pairs) > MAX_ISOVIST_VIEWPOINTS:
        step = max(1, len(face_centroid_pairs) // MAX_ISOVIST_VIEWPOINTS)
        sampled_pairs = face_centroid_pairs[::step][:MAX_ISOVIST_VIEWPOINTS]
        print(
            f"Isovist viewpoints reduced from {len(face_centroid_pairs)} to {len(sampled_pairs)} "
            f"using '{PERFORMANCE_PRESET}' preset."
        )
    else:
        sampled_pairs = face_centroid_pairs
        print(f"Using all {len(sampled_pairs)} isovist viewpoints.")

    iso_host_faces = [p[0] for p in sampled_pairs]
    iso_verts = [p[1] for p in sampled_pairs]

Manual point augmentation added: 30
Using manual viewpoints: 40


## 11. Spatial Intelligence through Isovists and Graph Analysis

## Visibility Graph Analysis
### Create Isovists
* Optimized with preset-driven sampling and host-face guidance.
* For faster runs, use `PERFORMANCE_PRESET = "fast"` in Section 7.
* If you need maximum coverage, use `PERFORMANCE_PRESET = "high"` (slower).

In [54]:
# Fast path: compute each isovist on a navigable face first.
# Avoid calling Topology.Faces on a Face to prevent warning spam.
gallery_type = Topology.TypeAsString(gallery) if gallery else None
if gallery_type == "Face":
    gallery_faces = [gallery]
else:
    gallery_faces = [f for f in (Topology.Faces(gallery) or []) if f]

if not gallery_faces:
    raise ValueError("No valid faces found in gallery for isovist computation.")

primary_iso_face = gallery_faces[0]

def _as_bool(x):
    if isinstance(x, list):
        return any(bool(v) for v in x)
    return bool(x)

isovists = []
valid_iso_verts = []
failed_count = 0
fallback_count = 0
repaired_viewpoints = 0

# Keep False unless you need recovery for edge-case points; full fallback is expensive.
ENABLE_FULL_FACE_FALLBACK = False

for host_face, v in zip(iso_host_faces, iso_verts):
    found = None
    vp = v

    # 1) Preferred: compute isovist on primary navigable face.
    inside_primary = _as_bool(Vertex.IsInternal2D(vp, primary_iso_face))
    if inside_primary:
        found = Face.Isovist(primary_iso_face, vp)

    # 2) If not valid, repair viewpoint to host-face centroid and retry on primary face.
    if not found and host_face:
        centroid_v = Topology.Centroid(host_face)
        if centroid_v:
            inside_centroid = _as_bool(Vertex.IsInternal2D(centroid_v, primary_iso_face))
            if inside_centroid:
                found = Face.Isovist(primary_iso_face, centroid_v)
                if found:
                    vp = centroid_v
                    repaired_viewpoints += 1

    # 3) Optional expensive fallback: scan all gallery faces.
    if not found and ENABLE_FULL_FACE_FALLBACK:
        for gf in gallery_faces:
            cand = Face.Isovist(gf, vp)
            if cand:
                found = cand
                fallback_count += 1
                break

    if found:
        isovists.append(found)
        valid_iso_verts.append(vp)
    else:
        failed_count += 1

print(
    f"Computed {len(isovists)} valid isovists out of {len(iso_verts)} viewpoints "
    f"(failed={failed_count}, repaired={repaired_viewpoints}, fallback_used={fallback_count})."
 )

Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible po

#### Show the computed isovists overlaid on the floor plan

In [62]:
if not isovists:
    print("No valid isovists to display.")
else:
    Topology.Show(gallery, isovists,
                  faceColorKey="cp_color",
                  faceOpacity=0.6,
                  showEdges=False,
                  showVertices=False,
                  camera=[0,0,6],
                  backgroundColor="black",
                  width=800,
                  height=600,
                  renderer=renderer)

### Compute the visibility of each isovist viewpoint
* Calculate how many other points of the dense grid are within each isovist's face.

In [56]:
new_verts = []
n_list = []

# Prefer graph vertices for stable sampling density; use grid1 only if substantially denser.
analysis_points = g_verts
grid_points = Topology.Vertices(grid1) if 'grid1' in globals() and grid1 else []
grid_points = [p for p in (grid_points or []) if p]
if len(grid_points) > (2 * len(g_verts)):
    analysis_points = grid_points
print(f"Visibility sampling points: {len(analysis_points)}")

if not isovists:
    print("No valid isovists found. Assigning visibility = 0 to all analysis vertices.")
    for v in g_verts:
        d = Topology.Dictionary(v)
        d = Dictionary.SetValueAtKey(d, "visibility", 0)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)
    n_list = [0]
else:
    for i, iso in enumerate(isovists):
        v = valid_iso_verts[i]
        b_list = Vertex.IsInternal2D(analysis_points, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)

print(f"Visibility stats: min={min(n_list)}, max={max(n_list)}, mean={sum(n_list)/len(n_list):.2f}")

Visibility sampling points: 611
Visibility stats: min=2, max=41, mean=10.94


### Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [57]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### Derive the colour of each vertex based on the interpolated value

In [58]:
# Robust color mapping: clip extremes and boost contrast so output is visually readable.
vis_values = []
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    if vb is not None:
        vis_values.append(float(vb))

if not vis_values:
    raise ValueError("No visibility values found on graph vertices.")

vis_values_sorted = sorted(vis_values)
n_vals = len(vis_values_sorted)

def _percentile(sorted_vals, p):
    if len(sorted_vals) == 1:
        return sorted_vals[0]
    idx = int((len(sorted_vals) - 1) * p)
    return sorted_vals[idx]

# Clip outliers for stronger visual contrast.
p_low = _percentile(vis_values_sorted, 0.05)
p_high = _percentile(vis_values_sorted, 0.95)

# Fallback if clipped range collapses.
if p_low == p_high:
    p_low = min(vis_values_sorted)
    p_high = max(vis_values_sorted)

# Gamma < 1 boosts mid-range contrast.
gamma = 0.75

for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    vb = float(vb) if vb is not None else p_low

    if p_high == p_low:
        norm = 0.5
    else:
        norm = (vb - p_low) / (p_high - p_low)
        norm = max(0.0, min(1.0, norm))

    norm = norm ** gamma
    color = Color.AnyToHex(
        Color.ByValueInRange(norm, minValue=0, maxValue=1, colorScale="thermal")
    )

    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    d = Dictionary.SetValueAtKey(d, "visibility_norm", norm)
    _ = Topology.SetDictionary(v, d)

print(
    f"Visibility range(raw): min={min(vis_values_sorted):.2f}, max={max(vis_values_sorted):.2f} | "
    f"range(visual): p05={p_low:.2f}, p95={p_high:.2f}, gamma={gamma}"
)

Visibility range(raw): min=2.00, max=41.00 | range(visual): p05=4.21, p95=17.27, gamma=0.75


### Transfer the information from the graph vertices to the faces of the original shell

In [59]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

#### Show the final visibility heatmap on the floor-plan faces

In [61]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)